# Load Packages

In [52]:
import os
import pandas as pd
import numpy as np
from patsy import dmatrix
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from custom_functions import _prior_mean
from brand_extraction import add_brand_column
# import matplotlib.pyplot as plt

# Parameters

In [53]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load Data

In [54]:
# Load data
toaster_sentiment_df = pd.read_csv(os.path.join(data_processed_dir, "toaster_sentiment.csv"))

# Preview data
display(toaster_sentiment_df.head())

toaster_sentiment_df.info()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,SRB_neg,SRB_neu,SRB_pos,SRB_CS,RVRB_SENT,RVRB_SCORE,RVRB_neg,RVRB_neu,RVRB_pos,RVRB_CS
0,B01KZ729F6,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,NaN,0.220000,NaN,1.0,0,4.4,12579.0,https://www.amazon.com/stores/HamiltonBeach/pa...,...,0.0012,0.0,0.9988,0.9976,positive,0.9982,0.0018,0.0,0.9982,0.9964
1,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.80,1.0,0,4.1,4156.0,https://www.amazon.com/stores/Nostalgia/page/B...,...,0.0018,0.0,0.9982,0.9964,positive,0.9938,0.0062,0.0,0.9938,0.9876
2,B0BT5WXBR2,Elite Gourmet ECT118B Cool Touch Single Slice ...,14.990000,0.000000,14.99,1.0,0,NaN,NaN,https://www.amazon.com/stores/EliteGourmet/pag...,...,0.9995,0.0,0.0005,-0.9990,negative,0.9986,0.9986,0.0,0.0014,-0.9972
3,B0B9MX21NV,"evoloop Toaster 2 Slice, Stainless Steel Bread...",279.850020,0.874969,34.99,1.0,0,4.4,31.0,https://www.amazon.com/stores/evoloop/page/087...,...,0.0022,0.0,0.9979,0.9957,negative,0.5023,0.5023,0.0,0.4977,-0.0046
4,B0721CGB5F,BLACK+DECKER 4-Slice Toaster Oven with Natural...,64.990000,0.000000,64.99,1.0,0,4.4,25902.0,https://www.amazon.com/stores/BLACKDECKER/page...,...,0.0011,0.0,0.9989,0.9978,positive,0.9986,0.0014,0.0,0.9986,0.9972


<class 'pandas.DataFrame'>
RangeIndex: 59586 entries, 0 to 59585
Data columns (total 50 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ASIN         59586 non-null  str    
 1   P_TITLE      59586 non-null  str    
 2   OP           43859 non-null  float64
 3   DP           59586 non-null  float64
 4   SP           43859 non-null  float64
 5   FS           57322 non-null  float64
 6   PRA_4.5      59586 non-null  int64  
 7   P_RTG        56119 non-null  float64
 8   RTG_P_NO     56119 non-null  float64
 9   SELLER_LINK  59586 non-null  str    
 10  IMAGE_URL    59586 non-null  str    
 11  P_URL        59586 non-null  str    
 12  RV_URL       59586 non-null  str    
 13  PRFL_IMG     59586 non-null  str    
 14  PRFL_URL     59586 non-null  str    
 15  RV_TTL       59584 non-null  str    
 16  RVS          59586 non-null  str    
 17  RVR          59583 non-null  str    
 18  RSR          59586 non-null  int64  
 19  RVR_CONT     59

## Preprocess Data

In [55]:
# Convert review date to datetime and extract quarter
toaster_sentiment_df["RV_DT"] = pd.to_datetime(toaster_sentiment_df["RV_DT"], errors="coerce")
toaster_sentiment_df["RV_QTR"] = toaster_sentiment_df["RV_DT"].dt.to_period("Q")
display(toaster_sentiment_df[["RV_DT", "RV_QTR"]].head())

,RV_DT,RV_QTR
0,2019-01-06,2019Q1
1,2018-12-09,2018Q4
2,2021-03-28,2021Q1
3,2022-11-17,2022Q4
4,2022-10-03,2022Q4


In [56]:
# Coerce numeric columns to numeric
num_cols = [
    "OP", "DP", "SP", "FS", "PRA_4.5", "P_RTG", "RTG_P_NO",
    "RSR", "VP", "HLP_VT", "IMG_PRST", "TTL_RV", "RVS_L",
    "SUBJ", "CP_RVS", "TWRB_SCORE", "SRB_SCORE", "RVRB_SCORE"]
    
for col in num_cols:
    toaster_sentiment_df[col] = pd.to_numeric(toaster_sentiment_df[col], errors="coerce") 

# Model Variables

In [57]:
# Display key variables
display(toaster_sentiment_df[["VP", "IMG_PRST", "RVS_L", "HLP_VT", "TTL_RV", "SUBJ", "RSR", "CP_RVS", "TWRB_CS", "SRB_CS", "RVRB_CS"]].head())

# Summary of key variables
display(toaster_sentiment_df[["VP", "IMG_PRST", "RVS_L", "HLP_VT", "TTL_RV", "SUBJ", "RSR", "CP_RVS", "TWRB_CS", "SRB_CS", "RVRB_CS"]].describe().round(3))

,VP,IMG_PRST,RVS_L,HLP_VT,TTL_RV,SUBJ,RSR,CP_RVS,TWRB_CS,SRB_CS,RVRB_CS
0,1,0,31.0,NaN,1669.0,0.600000,5,0.6369,0.9630,0.9976,0.9964
1,1,0,96.0,NaN,863.0,0.675000,5,0.8519,0.8806,0.9964,0.9876
2,1,0,79.0,NaN,4078.0,0.320000,3,0.4404,-0.6753,-0.9990,-0.9972
3,0,1,3565.0,1.0,61.0,0.509831,4,0.9914,-0.1489,0.9957,-0.0046
4,1,0,91.0,NaN,3977.0,1.000000,5,0.5719,0.9838,0.9978,0.9972


,VP,IMG_PRST,RVS_L,HLP_VT,TTL_RV,SUBJ,RSR,CP_RVS,TWRB_CS,SRB_CS,RVRB_CS
count,59586.000,59586.000,59586.000,13506.000,59525.000,59585.000,59586.000,59585.000,59585.000,59585.000,59585.000
mean,0.939,0.064,188.384,4.306,2602.501,0.545,3.763,0.457,0.317,0.303,0.286
std,0.239,0.244,233.728,22.230,2084.109,0.256,1.588,0.458,0.752,0.949,0.938
min,0.000,0.000,11.000,1.000,3.000,0.000,1.000,-0.970,-0.958,-0.999,-0.999
25%,1.000,0.000,54.000,1.000,716.000,0.421,2.000,0.060,-0.531,-0.999,-0.996
50%,1.000,0.000,117.000,1.000,2220.000,0.588,5.000,0.625,0.799,0.997,0.990
75%,1.000,0.000,236.000,3.000,3977.000,0.717,5.000,0.836,0.963,0.998,0.997
max,1.000,1.000,5979.000,1466.000,14944.000,1.000,5.000,1.000,0.990,0.998,0.998


## Sentiment Variable

In [58]:
# Choose a sentiment variable from the multiple sentiment measures available.
sentiment_var = "RVRB_CS"  # Change this to switch between sentiment measures (e.g., "TWRB_SC", "SRB_SC", "RVRB_SC")

# Standardize the chosen sentiment variable using z-score normalization, ignoring NaN values
toaster_sentiment_df["sentiment"] = zscore(toaster_sentiment_df[sentiment_var], nan_policy="omit")

# Remove rows with missing sentiment values after standardization
toaster_sentiment_df = toaster_sentiment_df.dropna(subset=["sentiment"]).copy()

# Centre sentiment around its mean to improve interpretability of regression coefficients
toaster_sentiment_df["sentiment_c"] = toaster_sentiment_df["sentiment"] - toaster_sentiment_df["sentiment"].mean()

## Interaction Terms

In [59]:
# Interaction between sentiment and verified purchase and image presence 
toaster_sentiment_df["vp_img"] = toaster_sentiment_df["IMG_PRST"] * toaster_sentiment_df["VP"]
toaster_sentiment_df["sent_vp"] = toaster_sentiment_df["sentiment_c"] * toaster_sentiment_df["VP"]
toaster_sentiment_df["sent_img"] = toaster_sentiment_df["sentiment_c"] * toaster_sentiment_df["IMG_PRST"]
toaster_sentiment_df["sent_vp_img"] = toaster_sentiment_df["sentiment_c"] * toaster_sentiment_df["VP"] * toaster_sentiment_df["IMG_PRST"]

## Control Variables

In [60]:
# Logarithm of review length
toaster_sentiment_df["log_length"] = np.log(toaster_sentiment_df["RVS_L"])

# Logarithm of helpful votes
toaster_sentiment_df["log_helpful"] = np.log(toaster_sentiment_df["HLP_VT"].fillna(0).astype(float) + 1)  # Add 1 to avoid log(0)

# Logarithm of total reviews
toaster_sentiment_df["log_reviews"] = np.log(toaster_sentiment_df["TTL_RV"].fillna(0).astype(float))  # Add 1 to avoid log(0)

# Prior mean rating; the average rating of product based on all reviews posted prior to a certain timestamp (e.g., review date)
toaster_sentiment_df = toaster_sentiment_df.sort_values(["ASIN", "RV_DT"]).copy()

toaster_sentiment_df["prior_mean_rating"] = (
    toaster_sentiment_df
    .groupby("ASIN", group_keys=False)[["RSR"]]
    .apply(_prior_mean)
)

# Remove first review per product (no prior info)
# This line can be ignored as it can handled when we drop NA values in the modelling dataframe (df_mod) later on. 
# toaster_sentiment_df = toaster_sentiment_df[toaster_sentiment_df["prior_mean_rating"].notna()].copy()

/Users/fbilsond/miniforge3/envs/rating-dist-env/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [61]:
# Create brand names from product titles, product URLs, and seller/store URLs
toaster_sentiment_df = add_brand_column(
    toaster_sentiment_df,
    brand_col="BRAND",
    source_cols=["P_TITLE", "P_URL", "SELLER_LINK"],
)

print(
    f"Brand missing after extraction: {toaster_sentiment_df['BRAND'].isna().sum():,} "
    f"rows out of {len(toaster_sentiment_df):,}"
)

Brand missing after extraction: 0 rows out of 59,585


In [62]:
# df["VP"]      = pd.Categorical(df["VP"].astype(str))
# df["IMG"]     = pd.Categorical(df["IMG_PRST"].astype(str))
toaster_sentiment_df["product"] = pd.Categorical(toaster_sentiment_df["ASIN"])
toaster_sentiment_df["brand"]   = pd.Categorical(toaster_sentiment_df["BRAND"])
toaster_sentiment_df["quarter"] = pd.Categorical(toaster_sentiment_df["RV_QTR"])


In [63]:
print(f"  Final modelling N = {len(toaster_sentiment_df):,}")
print(f"  Products: {toaster_sentiment_df['product'].nunique()};  "
      f"Brands: {toaster_sentiment_df['brand'].nunique()};  "
      f"Quarters: {toaster_sentiment_df['quarter'].nunique()}")

  Final modelling N = 59,585
  Products: 322;  Brands: 112;  Quarters: 21


# Stage 1: Baseline Model

In [64]:
# B-spline basis with 4 degrees of freedom)
# RSR ~ B-spline(sentiment, df=4)
# Residual = distortion (rating minus sentiment-predicted rating)
spline_basis = dmatrix(
    "bs(sentiment, df=4, include_intercept=False)",
    data=toaster_sentiment_df, 
    return_type="dataframe"
)

## OLS Fit to Compute Rating Distortion
RSR ~ spline(sentiment)

In [65]:
baseline_lm = LinearRegression().fit(spline_basis, toaster_sentiment_df["RSR"])
toaster_sentiment_df["pred_rating"] = baseline_lm.predict(spline_basis)

# Rating distortion = actual rating (RSR) minus predicted rating from sentiment
toaster_sentiment_df["distortion"] = toaster_sentiment_df["RSR"] - toaster_sentiment_df["pred_rating"]

print(f"Baseline R² = {baseline_lm.score(spline_basis, toaster_sentiment_df['RSR']):.4f}")
print(f"Distortion: mean = {toaster_sentiment_df['distortion'].mean():.4f}, "
      f"sd = {toaster_sentiment_df['distortion'].std():.4f}")

Baseline R² = 0.7966
Distortion: mean = 0.0000, sd = 0.7162


# Descriptive Statistics

In [66]:
# Final clean modelling frame
mod_cols = [
    # dependent variable
    "distortion",
    # platform cues variables
    "sentiment_c", "VP", "IMG_PRST",
    # interactions between sentiment and platform cues
    "vp_img", "sent_vp", "sent_img", "sent_vp_img",
    # controls      
    "prior_mean_rating", "log_length", "log_helpful", "log_reviews", "SUBJ",
    # grouping variables
    "quarter", "product", "brand"
]

df_mod = toaster_sentiment_df[mod_cols].dropna().reset_index(drop=True)

# df_mod["product_str"] = df_mod["product"].astype(str)
# df_mod["brand_str"]   = df_mod["brand"].astype(str)
# df_mod["quarter_str"] = df_mod["quarter"].astype(str)

print(f"  After final NA drop: N = {len(df_mod):,}")

  After final NA drop: N = 59,263


In [68]:
print(f"  Final modelling N = {len(df_mod):,}")
print(f"  Products: {df_mod['product'].nunique()}  "
      f"Brands: {df_mod['brand'].nunique()}  "
      f"Quarters: {df_mod['quarter'].nunique()}")

  Final modelling N = 59,263
  Products: 322  Brands: 112  Quarters: 21


In [67]:
# Descriptive statistics for numeric modelling columns
numeric_mod_cols = [
    col for col in mod_cols
    if pd.api.types.is_numeric_dtype(df_mod[col])
]

desc_source = df_mod[numeric_mod_cols].replace([np.inf, -np.inf], np.nan)
descriptive_stats = pd.DataFrame({
    "Variable": numeric_mod_cols,
    "mean": desc_source.mean(),
    "sd": desc_source.std(),
    "min": desc_source.min(),
    "median": desc_source.median(),
    "max": desc_source.max(),
}).reset_index(drop=True).round(3)

descriptive_stats_tex = os.path.join(
    output_dir,
    "modelling_column_descriptive_statistics.tex",
)
latex_table = descriptive_stats.to_latex(
    index=False,
    escape=True,
    column_format="lrrrrr",
    caption="Descriptive statistics for modelling variables",
    label="tab:modelling_descriptive_statistics",
)

with open(descriptive_stats_tex, "w") as f:
    f.write("% Requires \\usepackage{booktabs}\n")
    f.write(latex_table)

display(descriptive_stats)
print(f"Saved descriptive statistics table to {descriptive_stats_tex}")

,Variable,mean,sd,min,median,max
0,distortion,0.000,0.716,-3.922,0.115,3.281
1,sentiment_c,-0.001,1.000,-1.371,0.751,0.759
2,VP,0.940,0.238,0.000,1.000,1.000
3,IMG_PRST,0.063,0.244,0.000,0.000,1.000
4,vp_img,0.054,0.226,0.000,0.000,1.000
5,sent_vp,0.009,0.966,-1.371,0.748,0.759
6,sent_img,0.006,0.246,-1.371,0.000,0.759
7,sent_vp_img,0.003,0.230,-1.371,0.000,0.759
8,prior_mean_rating,3.892,0.471,1.000,3.817,5.000
9,log_length,4.724,1.036,2.398,4.762,8.696


Saved descriptive statistics table to ../outputs/modelling_column_descriptive_statistics.tex


# Mixed Linear Model (H1 Main Model)

In [ ]:
#  4. model_final — MIXED LINEAR MODEL  (H1 main model)
#     Equivalent to lmer(distortion ~ VP:IMG+sentiment_c*(VP+IMG)+controls +
#                        factor(quarter) + (1|product) + (1|brand), ...)
#
#     Implementation note:
#       statsmodels MixedLM supports one grouping factor natively.
#       Brand random effect is approximated by brand dummies (fixed) for the
#       main inference model, with a separate null model used for ICC.
#       Product is the primary grouping variable (random intercept).
# =============================================================================
print("\n[4/8]  Fitting model_final (MixedLM)...")

# quarter → FIXED  (C(quarter_str))
# product → RANDOM intercept  (groups=)
# brand   → RANDOM intercept  (vc_formula=)   ← corrected from previous version
formula_mlm = (
    "distortion ~ VP1 + IMG1 + VP_IMG + "
    "sent_VP + sent_IMG + sent_VP_IMG + "
    "prior_mean_rating + log_length + log_helpful + log_reviews + SUBJ + "
    "C(quarter_str)"
)

# REML=True matches R's default; powell optimizer mirrors bobyqa stability
mlm     = smf.mixedlm(
    formula_mlm, df_mod,
    groups     = df_mod["product_str"],
    vc_formula = {"brand": "0 + C(brand_str)"}   # brand as second RE
)
mlm_fit = mlm.fit(reml=True, method="powell", maxiter=500)
